In [23]:
import xmltodict
import re
import urllib.parse
import pandas as pd

import glob

import pprint as pp

In [25]:
filename_library_xml = '../data/rekordbox-2026-02-20.xml'
directory_track_location_root = 'file://localhost/Users/emily'

directory_local_track_location_root = '/media/emily/DJ_Backup'
local_cut = 'Desktop/DJ'

output_directory = 'output'

In [3]:
def load_xml_to_dict(filename):
    with open(filename, 'r') as xml_file:
        xml_content = xml_file.read()
        data_dictionary = xmltodict.parse(xml_content)
    return data_dictionary

In [4]:
dict_library = load_xml_to_dict(filename_library_xml)

In [5]:
def change_empty_string_to_None(dictionary):
    for key in dictionary.keys():
        if dictionary[key].strip() == '':
            dictionary[key] = None
    return dictionary

def is_float_regex(s):
    regex_pattern = r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$"
    return bool(re.match(regex_pattern, s))

def none_or_int(value):
    if is_float_regex(value):
        return int(value)
    else:
        return None

def none_or_float(value):
    if is_float_regex(value):
        return float(value)
    else:
        return None

In [6]:
list_of_entries = []
for entry in dict_library['DJ_PLAYLISTS']['COLLECTION']['TRACK']:

    entry_dict = {
        'album' : entry['@Album'],
        'artist' : entry['@Artist'],
        'genre' : entry['@Genre'],
        'composer' : entry['@Composer'],
        'name' : entry['@Name'],
        'tonality' : entry['@Tonality'],
    }

    entry_dict = change_empty_string_to_None(entry_dict)
    entry_dict['year'] = none_or_int(entry['@Year'])
    entry_dict['average_bpm'] = none_or_float(entry['@AverageBpm'])
    entry_dict['total_time'] = none_or_int(entry['@TotalTime'])
    entry_dict['rekordbox_track_id'] = none_or_int(entry['@TrackID'])

    entry_dict['location'] = urllib.parse.unquote(
        entry['@Location'].replace(directory_track_location_root + '/', ''),
        encoding = 'utf-8',
    )
    if entry_dict['location'].strip() == '':
        entry_dict['location'] == None

    dynamic = None
    if 'TEMPO' in entry:
        dynamic = False
        if str(type(entry['TEMPO'])) == "<class 'list'>":
            dynamic = True
    entry_dict['dynamic'] = dynamic

    comments = entry['@Comments']

    energy = None
    try:
        index_energy = comments.find('Energy')
        if index_energy >= 0:
            energy = int(comments[index_energy:].split('Energy ')[-1].split(' ')[0])
    except:
        pass
    entry_dict['energy'] = energy

    list_of_entries.append(entry_dict)

In [7]:
import os, sys
import django
os.environ['DJANGO_SETTINGS_MODULE'] = 'music.settings'
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = 'true'  # https://docs.djangoproject.com/en/4.1/topics/async/#async-safety
django.setup()

In [8]:
from rx_content.models import Track

In [9]:
obj_list = [Track(**data_dict) for data_dict in list_of_entries]
objs = Track.objects.bulk_create(obj_list)

In [10]:
test_obj = objs[-1]
print(test_obj.id)

5250


In [ ]:
# print(glob.glob(directory_local_track_location_root + '/' + entry_dict['location'].replace(local_cut, '')[1:]))

In [11]:
for node in dict_library['DJ_PLAYLISTS']['PLAYLISTS']['NODE']['NODE']:
    if node['@Name'] == 'HoB - Goth Night - BDSM - 2026-02-13':
        break

rekordbox_track_id_list = []
for entry in node['NODE']:
    for track in entry['TRACK']:
        rekordbox_track_id = int(track['@Key'])
        rekordbox_track_id_list.append(rekordbox_track_id)


print(len(rekordbox_track_id_list))

193


In [13]:
queryset = Track.objects.filter(rekordbox_track_id__in = rekordbox_track_id_list)

In [26]:
the_list = []
for q in queryset:
    the_list.append(
        {
            'id' : q.id,
            'path' : directory_local_track_location_root + '/' + q.location.replace(local_cut, '')[1:],
        }
    )
df_playlist = pd.DataFrame(the_list)
df_playlist.to_parquet(output_directory + '/playlist.parquet')